# 🔬 Análise Exploratória — Docentes UFRPE (CAPES)

**Sprint 1**: Análise da Relação entre Tempo de Carreira e Impacto Científico na UFRPE

**Grupo**: José Rafael e Stella | **Disciplina**: Gestão da Informação e do Conhecimento

---

### Objetivo deste notebook
1. Carregar e inspecionar os CSVs de docentes da CAPES (2017–2024)
2. Filtrar docentes da UFRPE
3. Avaliar qualidade dos dados (nulos, duplicatas, inconsistências)
4. Análise exploratória: distribuições, correlações, padrões temporais
5. Propor limpeza e normalização para alimentar o pipeline OML


## 1. Setup e Carregamento

In [ ]:
# Montar Google Drive (executar apenas no Colab)
from google.colab import drive
drive.mount('/content/Drive')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='viridis')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100

DATA_DIR = '/content/Drive/MyDrive/gic/data/raw'
print("✓ Setup concluído")

### 1.1 Leitura dos CSVs

In [ ]:
from pathlib import Path
import glob

csv_files = sorted(glob.glob(f'{DATA_DIR}/br-capes-colsucup-docente*.csv'))
print(f"Arquivos encontrados: {len(csv_files)}")
for f in csv_files:
    print(f"  • {Path(f).name}")

In [ ]:
# Carregar e concatenar todos os CSVs
dfs = []
for f in csv_files:
    df = pd.read_csv(f, delimiter=';', encoding='iso-8859-1', low_memory=False)
    dfs.append(df)
    print(f"  {Path(f).name}: {len(df):,} linhas")

df_raw = pd.concat(dfs, ignore_index=True)
print(f"\nTotal: {len(df_raw):,} linhas × {len(df_raw.columns)} colunas")

## 2. Inspeção Inicial

### 2.1 Colunas disponíveis

In [ ]:
# Lista de todas as colunas e seus tipos
col_info = pd.DataFrame({
    'tipo': df_raw.dtypes,
    'nulos': df_raw.isnull().sum(),
    'nulos_%': (df_raw.isnull().sum() / len(df_raw) * 100).round(2),
    'unicos': df_raw.nunique(),
    'exemplo': df_raw.iloc[0]
})
col_info

### 2.2 Primeiras linhas (amostra geral)

In [ ]:
df_raw.head(3)

### 2.3 Anos-base presentes

In [ ]:
print("Anos-base nos dados:")
print(sorted(df_raw['AN_BASE'].unique()))
print(f"\nRegistros por ano:")
df_raw['AN_BASE'].value_counts().sort_index()

## 3. Filtragem: Docentes da UFRPE

Vamos isolar apenas os registros da UFRPE e entender o universo de docentes.

In [ ]:
# Filtrar UFRPE
df_ufrpe = df_raw[df_raw['SG_ENTIDADE_ENSINO'].str.strip().str.upper() == 'UFRPE'].copy()
print(f"Registros UFRPE: {len(df_ufrpe):,} de {len(df_raw):,} ({len(df_ufrpe)/len(df_raw)*100:.2f}%)")
print(f"\nAnos-base UFRPE: {sorted(df_ufrpe['AN_BASE'].unique())}")
print(f"Docentes únicos (por nome): {df_ufrpe['NM_DOCENTE'].nunique():,}")

In [ ]:
# Registros por ano na UFRPE
ax = df_ufrpe['AN_BASE'].value_counts().sort_index().plot(kind='bar', color='teal', edgecolor='white')
ax.set_title('Registros de Docentes UFRPE por Ano-Base')
ax.set_xlabel('Ano')
ax.set_ylabel('Nº de registros')
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width()/2, p.get_height()),
                ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

## 4. Qualidade dos Dados

### 4.1 Valores nulos nas colunas-chave

In [ ]:
colunas_chave = [
    'NM_DOCENTE', 'AN_TITULACAO', 'IN_DOUTOR', 'NM_GRAU_TITULACAO',
    'DS_CATEGORIA_DOCENTE', 'NM_PROGRAMA_IES', 'NM_AREA_CONHECIMENTO',
    'CD_PROGRAMA_IES', 'AN_BASE'
]

nulos = df_ufrpe[colunas_chave].isnull().sum()
nulos_pct = (nulos / len(df_ufrpe) * 100).round(2)

pd.DataFrame({'nulos': nulos, '%': nulos_pct}).sort_values('nulos', ascending=False)

### 4.2 Valores do campo IN_DOUTOR

In [ ]:
print("Valores de IN_DOUTOR:")
print(df_ufrpe['IN_DOUTOR'].value_counts(dropna=False))
print(f"\nValores de NM_GRAU_TITULACAO:")
print(df_ufrpe['NM_GRAU_TITULACAO'].value_counts(dropna=False))

### 4.3 Categorias de docente

In [ ]:
print("DS_CATEGORIA_DOCENTE:")
print(df_ufrpe['DS_CATEGORIA_DOCENTE'].value_counts(dropna=False))

### 4.4 Duplicatas por nome + ano

In [ ]:
# Verificar se há docentes duplicados no mesmo ano
dupl = df_ufrpe.duplicated(subset=['NM_DOCENTE', 'AN_BASE'], keep=False)
print(f"Registros duplicados (mesmo nome + ano): {dupl.sum()}")
if dupl.sum() > 0:
    print("\nExemplos de duplicatas:")
    display(df_ufrpe[dupl].sort_values(['NM_DOCENTE', 'AN_BASE']).head(10)[
        ['NM_DOCENTE', 'AN_BASE', 'NM_PROGRAMA_IES', 'DS_CATEGORIA_DOCENTE']
    ])
    print("\n→ Provavelmente docentes vinculados a mais de um programa.")

## 5. Foco: Doutores da UFRPE

Para a análise de correlação tempo × citações, precisamos apenas dos doutores.

In [ ]:
# Filtrar apenas doutores
df_dout = df_ufrpe[df_ufrpe['IN_DOUTOR'].str.strip().str.upper() == 'S'].copy()
print(f"Docentes doutores (registros): {len(df_dout):,}")
print(f"Docentes doutores (únicos por nome): {df_dout['NM_DOCENTE'].nunique():,}")

# Filtrar permanentes + colaboradores
cats = ['PERMANENTE', 'COLABORADOR']
df_filt = df_dout[df_dout['DS_CATEGORIA_DOCENTE'].str.strip().str.upper().isin(cats)].copy()
print(f"\nApós filtro de categoria ({cats}):")
print(f"  Registros: {len(df_filt):,}")
print(f"  Docentes únicos: {df_filt['NM_DOCENTE'].nunique():,}")

## 6. Análise: Ano de Titulação (proxy de tempo de carreira)

In [ ]:
# Converter AN_TITULACAO para numérico
df_filt['AN_TITULACAO'] = pd.to_numeric(df_filt['AN_TITULACAO'], errors='coerce')

print(f"AN_TITULACAO — Estatísticas descritivas:")
print(df_filt['AN_TITULACAO'].describe().to_string())
print(f"\nNulos: {df_filt['AN_TITULACAO'].isnull().sum()}")

In [ ]:
# Distribuição do ano de titulação
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma
df_filt['AN_TITULACAO'].dropna().astype(int).hist(bins=30, ax=axes[0], color='teal', edgecolor='white')
axes[0].set_title('Distribuição do Ano de Doutorado')
axes[0].set_xlabel('Ano de Titulação')
axes[0].set_ylabel('Frequência')
axes[0].axvline(2020, color='red', linestyle='--', alpha=0.7, label='2020 (pandemia)')
axes[0].legend()

# Boxplot
df_filt.boxplot(column='AN_TITULACAO', ax=axes[1])
axes[1].set_title('Boxplot — Ano de Titulação')

plt.tight_layout()
plt.show()

In [ ]:
# Calcular tempo de carreira (anos desde doutorado) usando o AN_BASE como referência
df_filt['TEMPO_CARREIRA'] = df_filt['AN_BASE'] - df_filt['AN_TITULACAO']

print("TEMPO_CARREIRA (anos desde doutorado):")
print(df_filt['TEMPO_CARREIRA'].describe().to_string())
print(f"\nValores negativos (possível erro): {(df_filt['TEMPO_CARREIRA'] < 0).sum()}")
print(f"Valores > 50 (possível outlier): {(df_filt['TEMPO_CARREIRA'] > 50).sum()}")

In [ ]:
# Distribuição do tempo de carreira
fig, ax = plt.subplots(figsize=(12, 5))
df_filt['TEMPO_CARREIRA'].dropna().hist(bins=40, ax=ax, color='darkcyan', edgecolor='white')
ax.set_title('Distribuição do Tempo de Carreira (anos desde o doutorado)')
ax.set_xlabel('Anos')
ax.set_ylabel('Frequência')
ax.axvline(df_filt['TEMPO_CARREIRA'].median(), color='red', linestyle='--', label=f'Mediana: {df_filt["TEMPO_CARREIRA"].median():.0f} anos')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Análise por Área do Conhecimento

In [ ]:
# Top 15 áreas com mais docentes
top_areas = (df_filt.drop_duplicates(subset=['NM_DOCENTE'])
             .groupby('NM_AREA_CONHECIMENTO').size()
             .sort_values(ascending=True).tail(15))

fig, ax = plt.subplots(figsize=(12, 6))
top_areas.plot(kind='barh', ax=ax, color='teal', edgecolor='white')
ax.set_title('Top 15 Áreas do Conhecimento — Docentes Doutores UFRPE')
ax.set_xlabel('Nº de docentes únicos')
plt.tight_layout()
plt.show()

In [ ]:
# Tempo médio de carreira por área (top 10)
career_by_area = (df_filt.drop_duplicates(subset=['NM_DOCENTE'])
                  .groupby('NM_AREA_CONHECIMENTO')['TEMPO_CARREIRA']
                  .agg(['mean', 'median', 'count'])
                  .query('count >= 5')
                  .sort_values('mean', ascending=False)
                  .head(15))
career_by_area.columns = ['Média (anos)', 'Mediana (anos)', 'Nº docentes']
career_by_area.round(1)

## 8. Análise por Programa de Pós-Graduação

In [ ]:
# Programas com mais docentes doutores
progs = (df_filt.drop_duplicates(subset=['NM_DOCENTE', 'NM_PROGRAMA_IES'])
         .groupby('NM_PROGRAMA_IES').size()
         .sort_values(ascending=True).tail(15))

fig, ax = plt.subplots(figsize=(12, 6))
progs.plot(kind='barh', ax=ax, color='darkcyan', edgecolor='white')
ax.set_title('Top 15 Programas — Nº de Docentes Doutores UFRPE')
ax.set_xlabel('Nº de docentes')
plt.tight_layout()
plt.show()

## 9. Análise Temporal: Pré vs Pós-Pandemia

Quadriênio **pré-pandemia** (2017–2020) vs **pós-pandemia** (2021–2024).

In [ ]:
# Criar flag de período
df_filt['PERIODO'] = df_filt['AN_BASE'].apply(
    lambda x: 'Pré-Pandemia (2017–2020)' if x <= 2020 else 'Pós-Pandemia (2021–2024)'
)

# Contagem de docentes únicos por período
por_periodo = df_filt.groupby('PERIODO')['NM_DOCENTE'].nunique()
print("Docentes únicos por período:")
print(por_periodo.to_string())

In [ ]:
# Evolução anual de docentes únicos
evolucao = df_filt.groupby('AN_BASE')['NM_DOCENTE'].nunique()

fig, ax = plt.subplots(figsize=(12, 5))
evolucao.plot(kind='bar', ax=ax, color=['coral' if x <= 2020 else 'teal' for x in evolucao.index],
              edgecolor='white')
ax.set_title('Evolução Anual de Docentes Doutores Únicos — UFRPE')
ax.set_xlabel('Ano')
ax.set_ylabel('Docentes únicos')
ax.axvline(3.5, color='gray', linestyle='--', alpha=0.5)
ax.annotate('← Pré-pandemia | Pós-pandemia →', xy=(3.5, ax.get_ylim()[1]*0.95),
            ha='center', fontsize=10, color='gray')
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width()/2, p.get_height()),
                ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Comparação de tempo de carreira entre períodos
fig, ax = plt.subplots(figsize=(10, 5))
df_filt.boxplot(column='TEMPO_CARREIRA', by='PERIODO', ax=ax)
ax.set_title('Tempo de Carreira por Período')
plt.suptitle('')
ax.set_ylabel('Anos desde o doutorado')
plt.tight_layout()
plt.show()

## 10. Deduplicação: Construindo a tabela final de docentes

Para a ontologia, precisamos de **1 registro por docente** com os dados mais recentes.

In [ ]:
# Deduplicar: manter registro mais recente por docente
df_dedup = (df_filt
            .sort_values('AN_BASE', ascending=False)
            .drop_duplicates(subset=['NM_DOCENTE'], keep='first')
            .reset_index(drop=True))

print(f"Docentes únicos após deduplicação: {len(df_dedup):,}")
print(f"\nColunas relevantes para a ontologia:")
colunas_onto = ['NM_DOCENTE', 'AN_TITULACAO', 'TEMPO_CARREIRA', 'NM_AREA_CONHECIMENTO',
                'NM_PROGRAMA_IES', 'DS_CATEGORIA_DOCENTE', 'AN_BASE']
df_dedup[colunas_onto].head(10)

In [ ]:
# Resumo estatístico dos docentes deduplicados
print("=== RESUMO FINAL ===")
print(f"Total de docentes: {len(df_dedup)}")
print(f"AN_TITULACAO — min: {df_dedup['AN_TITULACAO'].min():.0f}, max: {df_dedup['AN_TITULACAO'].max():.0f}")
print(f"TEMPO_CARREIRA — mediana: {df_dedup['TEMPO_CARREIRA'].median():.0f} anos")
print(f"Áreas: {df_dedup['NM_AREA_CONHECIMENTO'].nunique()}")
print(f"Programas: {df_dedup['NM_PROGRAMA_IES'].nunique()}")
print(f"\nNulos em AN_TITULACAO: {df_dedup['AN_TITULACAO'].isnull().sum()}")
print(f"Tempo de carreira negativo: {(df_dedup['TEMPO_CARREIRA'] < 0).sum()}")

## 11. Anomalias e Limpeza Sugerida

In [ ]:
# Identificar registros problemáticos
print("=== ANOMALIAS DETECTADAS ===\n")

# 1) AN_TITULACAO nulo
n_null = df_dedup['AN_TITULACAO'].isnull().sum()
print(f"1) AN_TITULACAO nulo: {n_null} docentes")

# 2) Tempo de carreira negativo
neg = df_dedup[df_dedup['TEMPO_CARREIRA'] < 0]
print(f"2) Tempo de carreira negativo: {len(neg)} docentes")
if len(neg) > 0:
    display(neg[['NM_DOCENTE', 'AN_TITULACAO', 'AN_BASE', 'TEMPO_CARREIRA']].head())

# 3) Tempo de carreira > 50 anos
old = df_dedup[df_dedup['TEMPO_CARREIRA'] > 50]
print(f"3) Tempo de carreira > 50 anos: {len(old)} docentes")
if len(old) > 0:
    display(old[['NM_DOCENTE', 'AN_TITULACAO', 'AN_BASE', 'TEMPO_CARREIRA']].head())

# 4) AN_TITULACAO muito antigo (antes de 1960)
antigo = df_dedup[df_dedup['AN_TITULACAO'] < 1960]
print(f"4) AN_TITULACAO < 1960: {len(antigo)} docentes")

# 5) Nomes possivelmente duplicados (variações)
from difflib import SequenceMatcher
nomes = df_dedup['NM_DOCENTE'].sort_values().tolist()
similares = []
for i in range(len(nomes)-1):
    ratio = SequenceMatcher(None, nomes[i], nomes[i+1]).ratio()
    if 0.85 < ratio < 1.0:
        similares.append((nomes[i], nomes[i+1], round(ratio, 3)))
print(f"\n5) Nomes muito similares (possíveis duplicatas): {len(similares)}")
if similares:
    for a, b, r in similares[:10]:
        print(f"   {r:.0%}  '{a}'  ↔  '{b}'")

## 12. Exportar dados limpos

Salvar a tabela deduplicada para uso no pipeline OML.

In [ ]:
# Exportar CSV limpo
output_path = '/content/Drive/MyDrive/gic/data/processed/docentes_ufrpe_limpo.csv'

# Remover anomalias críticas antes de exportar
df_clean = df_dedup[
    (df_dedup['AN_TITULACAO'].notna()) &
    (df_dedup['TEMPO_CARREIRA'] >= 0) &
    (df_dedup['TEMPO_CARREIRA'] <= 60)
].copy()

print(f"Registros removidos na limpeza: {len(df_dedup) - len(df_clean)}")
print(f"Docentes no dataset final: {len(df_clean)}")

# Salvar
import os
os.makedirs(os.path.dirname(output_path), exist_ok=True)
df_clean.to_csv(output_path, sep=';', encoding='utf-8-sig', index=False)
print(f"\n✓ Salvo em: {output_path}")

## 13. Próximos passos

- [ ] Enriquecer com dados do **Scopus** (citações, h-index) via cruzamento por nome
- [ ] Rodar o pipeline `generate_oml_docentes_ufrpe.py` sobre o CSV limpo
- [ ] Análise de correlação: **Tempo de Carreira × Citações**
- [ ] Comparação estatística entre os quadriênios pré e pós-pandemia
